# Evaluation

## Goal
Evaluate translations and connect technical metrics to user-facing quality.

## Setup
Select **Python (machine-translation-lab)** or the project `.venv` kernel. Run Setup, SacreBLEU, and the metric callback before Step 3. The Step 3 preparation cells load the model and data when executed; the training call remains commented out.

In [18]:
# Environment, files, and timing
import sys
from pathlib import Path
from time import perf_counter

# Data and tables
import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display


# Training and tokenization
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

print(sys.executable)
print("Setup imports complete. Run the SacreBLEU cell next.")

c:\Users\admin\Documents\Codex\2026-10-12\files-pasted-by-the-user-task\outputs\machine-translation-lab\.venv\Scripts\python.exe
Setup imports complete. Run the SacreBLEU cell next.


## Steps

### 1. SacreBLEU
The tutorial uses `evaluate.load("sacrebleu")`, which may fetch a small metric module. The installed `sacrebleu` package also supports local evaluation.

In [19]:
# TODO: Load a metric and inspect its expected inputs. Record the score scale and settings. 

# Local translation metric
from sacrebleu.metrics import BLEU
metric = BLEU()

### 2. Reference translations
Align each prediction with its references. Hugging Face Evaluate uses references grouped by example; direct SacreBLEU uses one sequence per reference stream.

In [20]:
# TODO: Prepare aligned predictions/references and calculate a score using your chosen API. 

import numpy as np


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    # Trainer pads generated batches with -100; it is not a valid token ID.
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    # Local SacreBLEU expects one list per reference stream.
    score = metric.corpus_score(decoded_preds, [decoded_labels])
    result = {"bleu": score.score}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

### 3. Training
Fine-tune T5-small after defining the metric callback above, following the [Hugging Face training section](https://huggingface.co/docs/transformers/tasks/translation#train). Run the model-loading cell, data-preparation cell, and trainer-configuration cell in order. Training begins when you uncomment and run `trainer.train()` below.

Each notebook has its own variables. The data-preparation cell recreates tokenization here, using up to 128 training examples and 32 validation examples for a small CPU experiment. It reserves a separate test split for final assessment. Loading may download the dataset or model if they are not cached.

In [21]:
# Load the tokenizer and model once in this notebook.
checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

#### Prepare training and validation data
This cell creates `tokenized_books` in this notebook. It contains the `train` and `validation` keys used by the trainer. The separate `books["test"]` split is held back from training and validation.

In [22]:
# Recreate the data in this notebook's kernel.
from datasets import DatasetDict

books = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr",
    cache_dir=str(Path.home() / ".cache" / "huggingface" / "datasets"),
)

# Reproducible 80/10/10 train/validation/test split.
train_holdout = books["train"].train_test_split(test_size=0.2, seed=42)
holdout = train_holdout["test"].train_test_split(test_size=0.5, seed=42)
books = DatasetDict({
    "train": train_holdout["train"],
    "validation": holdout["train"],
    "test": holdout["test"],
})

# Small subsets for the first CPU experiment; test data stays untouched.
train_limit = 128
validation_limit = 32
training_books = DatasetDict({
    "train": books["train"].select(range(min(train_limit, len(books["train"])))),
    "validation": books["validation"].select(
        range(min(validation_limit, len(books["validation"])))
    ),
})

prefix = "translate English to French: "

def preprocess_function(examples):
    inputs = [prefix + pair["en"] for pair in examples["translation"]]
    targets = [pair["fr"] for pair in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True)

tokenized_books = training_books.map(
    preprocess_function,
    batched=True,
    remove_columns=training_books["train"].column_names,
)
print(tokenized_books)


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 128
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 32
    })
})


#### Configure and build the trainer
These starter settings use CPU, small batches, and generated predictions for SacreBLEU. `compute_metrics` needs the tokenizer created above and the local `metric = BLEU()` instance. Evaluate the same validation examples before and after training.

In [23]:
# Run the model-loading and data-preparation cells above first.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = project_root / "outputs" / "t5-small-en-fr"
training_args = Seq2SeqTrainingArguments(
    output_dir=str(output_dir),
    use_cpu=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=False,
    bf16=False,
    push_to_hub=False,
    report_to="none",
    seed=42,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_books["train"],
    eval_dataset=tokenized_books["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

#### Train, compare, and save
Record validation loss and BLEU before and after training. A tiny run verifies the workflow; it does not establish translation quality. Save both model and tokenizer, then use the saved folder as the checkpoint in notebook 4.

In [24]:
# TODO: Run once you are ready to start the training experiment.
# Use the latest callback after rerunning the Reference translations cell.
trainer.compute_metrics = compute_metrics
baseline_metrics = trainer.evaluate()
train_result = trainer.train()
trained_metrics = trainer.evaluate()
display(pd.DataFrame([baseline_metrics, trained_metrics],
                     index=["before_training", "after_training"]))
checkpoint_dir = output_dir / "final"
trainer.save_model(str(checkpoint_dir))
tokenizer.save_pretrained(str(checkpoint_dir))
# Record the data split, example counts, settings, elapsed time, and observations.

Training Loss,Validation Loss,Epoch,Bleu,Gen Len
No log,2.093639,0,16.511600,52.218800


Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,2.065281,15.885300,52.312500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Bleu,Gen Len
No log,2.065281,1,15.885300,52.312500


,eval_loss,eval_bleu,eval_gen_len
before_training,2.093639,16.5116,52.2188
after_training,2.065281,15.8853,52.3125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('c:\\Users\\admin\\Documents\\Codex\\2026-10-12\\files-pasted-by-the-user-task\\outputs\\machine-translation-lab\\outputs\\t5-small-en-fr\\final\\tokenizer_config.json',
 'c:\\Users\\admin\\Documents\\Codex\\2026-10-12\\files-pasted-by-the-user-task\\outputs\\machine-translation-lab\\outputs\\t5-small-en-fr\\final\\tokenizer.json')

### 4. Error analysis
Run after Step 3. Inspect a bounded validation sample using the current trainer model. The table ranks examples for human review; digit, copying, and length checks are heuristic. Sentence BLEU is a diagnostic, and alternative correct translations may score poorly.

Check source/reference language alignment before attributing a low score to the model.

In [2]:
# Inspect a small validation sample from the model currently in the trainer.
import re
from collections import Counter
from decimal import Decimal
import torch
from sacrebleu.metrics import BLEU

analysis_model = trainer.model
analysis_size = min(10, len(training_books["validation"]))
if analysis_size == 0:
    raise ValueError("The validation split is empty. Run data preparation first.")
analysis_pairs = training_books["validation"].select(range(analysis_size))
sentence_metric = BLEU(effective_order=True)
generation_settings = {"max_new_tokens": 128, "max_length": None,
                       "num_beams": 1, "do_sample": False}


def number_tokens(text):
    # Simple digit check: accepts decimal commas/dots, but not number words
    # or thousands separators. Treat differences as prompts for human review.
    return Counter(Decimal(n.replace(",", "."))
                   for n in re.findall(r"\d+(?:[.,]\d+)?", text))


def translate_one(text):
    device = next(analysis_model.parameters()).device
    was_training = analysis_model.training
    analysis_model.eval()
    try:
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        started = perf_counter()
        prompt = "translate English to French: " + text
        input_truncated = len(tokenizer.encode(prompt)) > 128
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=128).to(device)
        with torch.inference_mode():
            generated = analysis_model.generate(**inputs, **generation_settings)
        prediction = tokenizer.decode(generated[0], skip_special_tokens=True).strip()
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        return prediction, perf_counter() - started, input_truncated
    finally:
        analysis_model.train(was_training)


rows = []
for example in analysis_pairs:
    source, reference = example["translation"]["en"], example["translation"]["fr"]
    prediction, seconds, truncated = translate_one(source)
    length_ratio = len(prediction.split()) / max(1, len(reference.split()))
    flags = []
    if not prediction:
        flags.append("empty output")
    if number_tokens(source) != number_tokens(prediction):
        flags.append("check numbers")
    if prediction.casefold() == source.strip().casefold():
        flags.append("copied source")
    if length_ratio < 0.5 or length_ratio > 2.0:
        flags.append("unusual length")
    if truncated:
        flags.append("input truncated")
    rows.append({
        "example_id": example["id"], "source": source, "reference": reference,
        "prediction": prediction,
        "sentence_bleu": sentence_metric.sentence_score(prediction, [reference]).score,
        "flags": "; ".join(flags) or "none detected",
        "review_notes": "",  # Record omissions, additions, fluency, or terminology errors.
    })

error_table = pd.DataFrame(rows).sort_values("sentence_bleu").reset_index(drop=True)
print(f"Reviewing {analysis_size} validation examples; showing the 5 lowest BLEU scores.")
print("Sentence BLEU ranks examples for inspection; flags are not confirmed errors.")
display(error_table.head(5).style.format({"sentence_bleu": "{:.2f}"}).set_properties(
    subset=["source", "reference", "prediction"],
    **{"white-space": "normal", "text-align": "left", "min-width": "160px", "max-width": "300px"}
))
print("The full sample is in error_table. Inspect source, reference, and prediction together.")


Reviewing 10 validation examples; showing the 5 lowest BLEU scores.
Sentence BLEU ranks examples for inspection; flags are not confirmed errors.


,example_id,source,reference,prediction,sentence_bleu,flags,review_notes
0,76401,Ce jour-là il avait horreur de parler.,That day he had a horror of speaking.,Ce jour-là il avait horreur de parler.,4.87,copied source,
1,65169,GREAT SURPRISE ON THE PART OF THE AUDIENCE.,Grand étonnement dans l’auditoire.,PRINCIPAUX SUR LA PARTIE DE LA VÉRIFICATION.,5.52,none detected,
2,112840,"When he arrived in Holland his provision failed him; but having heard that the inhabitants of that country were all rich and Christians, he made himself sure of being treated by them in the same manner as the Baron's castle, before he had been driven thence through the power of Miss Cunegund's bright eyes.","Ses provisions lui manquèrent quand il fut en Hollande; mais ayant entendu dire que tout le monde était riche dans ce pays-là, et qu'on y était chrétien, il ne douta pas qu'on ne le traitât aussi bien qu'il l'avait été dans le château de M. le baron, avant qu'il en eût été chassé pour les beaux yeux de mademoiselle Cunégonde.","Lorsqu'il est arrivé aux Pays-Bas, sa disposition ne lui a pas été échouée; mais après avoir entendu que les habitants de ce pays étaient tous riches et chrétiens, il s'est assuré qu'ils étaient traités de la même manière que le château du baron, avant qu'il ne soit dirigé de là par la puissance des yeux clairs de Mme Cunegund.",6.46,none detected,
3,50788,"""That's another pair of shoes!",-- C’est autre chose!,« C'est une autre paire de chaussures!,6.57,none detected,
4,5210,We had better not mention it.,"N’en parlons pas, cela vaut mieux.",Nous ne l'avons pas mentionné.,6.92,none detected,


The full sample is in error_table. Inspect source, reference, and prediction together.


### 5. Business-oriented translation quality
Run Step 4 first to define the generation helper. These six authored retail examples illustrate checks for numbers, units, brands, currency, and policy meaning. They are a small exercise set, not representative business evidence. Review the outputs, then enter your judgments and any agreed latency/cost assumptions in the decision cell.

In [3]:
# Small, authored retail examples for this exercise, not a production benchmark.
# Each alternatives list describes one term/concept expected in the French output.
retail_cases = [
    {"case_id": "quantity", "source": "Buy 2 bottles of water.",
     "reference": "Achetez 2 bouteilles d'eau.", "required_terms": [["bouteilles"]]},
    {"case_id": "unit", "source": "This bag contains 5 kg of rice.",
     "reference": "Ce sac contient 5 kg de riz.", "required_terms": [["kg", "kilogrammes"]]},
    {"case_id": "brand", "source": "This LEGO set contains 120 pieces.",
     "reference": "Cet ensemble LEGO contient 120 pièces.", "required_terms": [["LEGO"]]},
    {"case_id": "return_policy", "source": "Return unopened items within 30 days.",
     "reference": "Retournez les articles non ouverts dans un délai de 30 jours.",
     "required_terms": [["jours"], ["non ouverts", "non ouvert", "scellés", "scellé"]]},
    {"case_id": "price", "source": "The price is 12.50 euros.",
     "reference": "Le prix est de 12,50 euros.", "required_terms": [["euros", "euro", "EUR", "€"]]},
    {"case_id": "negation", "source": "Do not use this product on wet surfaces.",
     "reference": "N'utilisez pas ce produit sur des surfaces mouillées.",
     "required_terms": [["pas", "jamais", "ne pas"]]},
]


def contains_term(text, term):
    return re.search(r"(?<!\w)" + re.escape(term.casefold()) + r"(?!\w)",
                     text.casefold()) is not None


# Warm up once outside the timing sample; measure one request at a time.
_ = translate_one(retail_cases[0]["source"])
retail_rows = []
for case in retail_cases:
    prediction, seconds, truncated = translate_one(case["source"])
    terms_ok = all(any(contains_term(prediction, term) for term in alternatives)
                   for alternatives in case["required_terms"])
    retail_rows.append({
        "case_id": case["case_id"], "source": case["source"],
        "reference": case["reference"], "prediction": prediction,
        "nonempty": bool(prediction),
        "numbers_match": number_tokens(case["source"]) == number_tokens(prediction),
        "terms_match": terms_ok, "input_truncated": truncated,
        "latency_seconds": seconds,
    })
retail_review = pd.DataFrame(retail_rows)
retail_review["automatic_checks_pass"] = (
    retail_review["nonempty"] & retail_review["numbers_match"]
    & retail_review["terms_match"] & ~retail_review["input_truncated"]
)
retail_bleu = BLEU().corpus_score(
    retail_review["prediction"].tolist(), [retail_review["reference"].tolist()]
).score
display(retail_review[["case_id", "source", "reference", "prediction",
                       "numbers_match", "terms_match", "automatic_checks_pass"]].style.set_properties(
    subset=["source", "reference", "prediction"],
    **{"white-space": "normal", "text-align": "left", "min-width": "160px", "max-width": "300px"}
))
print("Term checks are heuristics; review synonyms, meaning, negation, units, and omissions.")


,case_id,source,reference,prediction,numbers_match,terms_match,automatic_checks_pass
0,quantity,Buy 2 bottles of water.,Achetez 2 bouteilles d'eau.,Achetez 2 bouteilles d'eau.,True,True,True
1,unit,This bag contains 5 kg of rice.,Ce sac contient 5 kg de riz.,Ce sac contient 5 kg de riz.,True,True,True
2,brand,This LEGO set contains 120 pieces.,Cet ensemble LEGO contient 120 pièces.,Ce jeu LEGO contient 120 pièces.,True,True,True
3,return_policy,Return unopened items within 30 days.,Retournez les articles non ouverts dans un délai de 30 jours.,Retournez les articles non ouverts dans les 30 jours.,True,True,True
4,price,The price is 12.50 euros.,"Le prix est de 12,50 euros.","Le prix est de 12,50 euros.",True,True,True
5,negation,Do not use this product on wet surfaces.,N'utilisez pas ce produit sur des surfaces mouillées.,Ne pas utiliser ce produit sur des surfaces humides.,True,True,True


Term checks are heuristics; review synonyms, meaning, negation, units, and omissions.


#### Review and decision
Fill in `human_review` after reading the retail translations. Rerun this cell to update the decision without generating translations again. Targets and cost assumptions remain unset until you supply them.

In [4]:
# Edit these after reviewing the translations, then rerun this cell only.
# None = unreviewed; True = acceptable; False = needs correction.
human_review = {case["case_id"]: None for case in retail_cases}
# Example: human_review["quantity"] = True
latency_budget_seconds = None  # Set an agreed per-request target, if available.
hourly_compute_cost_usd = None  # Optional infrastructure cost assumption.

passed = int(retail_review["automatic_checks_pass"].sum())
mean_seconds = float(retail_review["latency_seconds"].mean())
max_seconds = float(retail_review["latency_seconds"].max())
reviewed = sum(value is not None for value in human_review.values())
summary = pd.DataFrame([
    {"measure": "Retail cases", "value": len(retail_review)},
    {"measure": "Automatic checks passed", "value": passed},
    {"measure": "Human-reviewed cases", "value": reviewed},
    {"measure": "Retail corpus BLEU (0–100)", "value": round(retail_bleu, 2)},
    {"measure": "Mean warm request latency (seconds)", "value": round(mean_seconds, 3)},
    {"measure": "Maximum observed latency (seconds)", "value": round(max_seconds, 3)},
])
display(summary)
print("Timing includes tokenization, generation, and decoding on this machine.")
print("Six requests are a smoke check; they do not estimate production tail latency.")

if hourly_compute_cost_usd is not None:
    if hourly_compute_cost_usd < 0:
        raise ValueError("The hourly compute cost must be nonnegative.")
    cost_per_1000 = mean_seconds * 1000 / 3600 * hourly_compute_cost_usd
    print(f"Illustrative compute cost per 1,000 sequential requests: USD {cost_per_1000:.2f}")
    print("Assumes continuously billed compute; excludes idle time and service overhead.")

reasons = []
if passed < len(retail_review):
    reasons.append(f"{len(retail_review) - passed} cases need review of automatic flags")
if not all(human_review[case["case_id"]] is True for case in retail_cases):
    reasons.append("human review is incomplete or has rejected translations")
if latency_budget_seconds is None:
    reasons.append("a latency target has not been set")
elif latency_budget_seconds <= 0:
    raise ValueError("The latency budget must be positive.")
elif max_seconds > latency_budget_seconds:
    reasons.append("the maximum observed latency exceeds the chosen target")

release_decision = "HOLD — " + "; ".join(reasons) if reasons else "READY FOR A LARGER PILOT"
print(release_decision)
print("Production release also needs representative retail data, bilingual review, "
      "agreed quality/cost targets, and a load test. Book-domain BLEU alone is insufficient.")


,measure,value
0,Retail cases,6.000
1,Automatic checks passed,6.000
2,Human-reviewed cases,0.000
3,Retail corpus BLEU (0–100),72.560
4,Mean warm request latency (seconds),0.114
5,Maximum observed latency (seconds),0.146


Timing includes tokenization, generation, and decoding on this machine.
Six requests are a smoke check; they do not estimate production tail latency.
HOLD — human review is incomplete or has rejected translations; a latency target has not been set
Production release also needs representative retail data, bilingual review, agreed quality/cost targets, and a load test. Book-domain BLEU alone is insufficient.


## Checks
Run this cell after Steps 3–5. It checks the existing data, metric callback, analysis tables, and review decision without retraining. A failure names what needs attention. Passing checks confirms notebook consistency; bilingual review and representative retail testing remain necessary.

In [5]:
# Run after Steps 3–5. This cell does not train or generate translations.
required = ["books", "training_books", "tokenized_books", "tokenizer", "compute_metrics",
            "error_table", "analysis_size", "retail_cases", "retail_review", "retail_bleu",
            "human_review", "latency_budget_seconds", "summary", "release_decision"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Run the earlier notebook cells first. Missing: " + ", ".join(missing))

check_rows = []
def record_check(name, condition, detail):
    check_rows.append({"check": name, "status": "PASS" if condition else "FAIL", "detail": detail})

# IDs check row overlap; they do not rule out duplicate text or incorrect language labels.
split_ids = {name: set(books[name]["id"]) for name in ("train", "validation", "test")}
record_check("Separate data splits",
             all(split_ids[a].isdisjoint(split_ids[b])
                 for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]),
             "No example IDs shared between train, validation, and test.")
for split in ("train", "validation"):
    record_check(f"{split}: subset and tokenization",
                 set(training_books[split]["id"]).issubset(split_ids[split])
                 and 0 < len(training_books[split]) == len(tokenized_books[split])
                 and {"input_ids", "attention_mask", "labels"}.issubset(tokenized_books[split].column_names),
                 f"{len(tokenized_books[split])} tokenized rows; required columns present.")

# Regression check: generated IDs AND labels can contain Trainer's -100 padding.
probe_ids = np.array([tokenizer.encode("Ceci est une phrase de test suffisamment longue.")])
probe_padded = np.pad(probe_ids, ((0, 0), (0, 2)), constant_values=-100)
try:
    probe_clean = compute_metrics((probe_ids, probe_ids.copy()))
    probe_result = compute_metrics(((probe_padded, None), probe_padded.copy()))
    padding_ok = (all(np.isfinite(list(probe_result.values())))
                  and np.isclose(probe_result["bleu"], 100.0, atol=1e-4)
                  and probe_result == probe_clean)
    padding_detail = str(probe_result)
except Exception as exc:
    padding_ok, padding_detail = False, f"{type(exc).__name__}: {exc}"
record_check("Safe decoding of -100 padding", padding_ok, padding_detail)

validation_pairs = {item["id"]: item["translation"] for item in training_books["validation"]}
record_check("Error-analysis rows aligned",
             len(error_table) == analysis_size and error_table["example_id"].is_unique
             and all(row.example_id in validation_pairs
                     and row.source == validation_pairs[row.example_id]["en"]
                     and row.reference == validation_pairs[row.example_id]["fr"]
                     for row in error_table.itertuples()),
             "Sources and references match the sampled validation IDs.")
record_check("Sentence BLEU range", error_table["sentence_bleu"].between(0, 100.00001).all(),
             "All sentence scores are finite and between 0 and 100.")
cases_by_id = {case["case_id"]: case for case in retail_cases}
record_check("Retail rows aligned",
             len(retail_review) == len(retail_cases) and retail_review["case_id"].is_unique
             and set(retail_review["case_id"]) == set(cases_by_id)
             and all(row.source == cases_by_id[row.case_id]["source"]
                     and row.reference == cases_by_id[row.case_id]["reference"]
                     for row in retail_review.itertuples()),
             "One prediction per retail case with its matching source and reference.")
record_check("Retail scores and timing",
             np.isfinite(retail_bleu) and 0 <= retail_bleu <= 100.00001
             and np.isfinite(retail_review["latency_seconds"]).all()
             and retail_review["latency_seconds"].gt(0).all(),
             "Corpus BLEU is in range and measured request times are positive.")
expected_pass = (retail_review["nonempty"] & retail_review["numbers_match"]
                 & retail_review["terms_match"] & ~retail_review["input_truncated"])
record_check("Automatic flags consistent", retail_review["automatic_checks_pass"].equals(expected_pass),
             "A case passes only when all four automatic conditions pass.")
review_values_ok = (set(human_review) == set(cases_by_id)
                    and all(value is None or type(value) is bool for value in human_review.values()))
record_check("Human-review entries", review_values_ok, "Each case has None, True, or False.")
summary_values = summary.set_index("measure")["value"]
record_check("Summary counts current",
             summary_values.get("Retail cases") == len(retail_review)
             and summary_values.get("Automatic checks passed") == int(expected_pass.sum())
             and summary_values.get("Human-reviewed cases") == sum(v is not None for v in human_review.values()),
             "Rerun the review-and-decision cell after changing review inputs.")
ready_for_pilot = (review_values_ok and bool(expected_pass.all())
                   and all(human_review.get(case_id) is True for case_id in cases_by_id)
                   and latency_budget_seconds is not None and latency_budget_seconds > 0
                   and retail_review["latency_seconds"].max() <= latency_budget_seconds)
record_check("Decision matches review inputs",
             release_decision == "READY FOR A LARGER PILOT" if ready_for_pilot
             else release_decision.startswith("HOLD"),
             "Incomplete/rejected review or an unmet/unset latency target requires HOLD.")

checks_table = pd.DataFrame(check_rows)
display(checks_table)
failed_checks = checks_table.loc[checks_table["status"] == "FAIL", "check"].tolist()
if failed_checks:
    raise AssertionError("Fix these checks, then rerun: " + "; ".join(failed_checks))
print(f"All {len(checks_table)} notebook checks passed.")
print("These check notebook consistency, not translation quality or production readiness.")
print("Still review language alignment, duplicate text, translation meaning, and representative retail data.")


,check,status,detail
0,Separate data splits,PASS,"No example IDs shared between train, validatio..."
1,train: subset and tokenization,PASS,128 tokenized rows; required columns present.
2,validation: subset and tokenization,PASS,32 tokenized rows; required columns present.
3,Safe decoding of -100 padding,PASS,"{'bleu': 100.0, 'gen_len': np.float64(11.0)}"
4,Error-analysis rows aligned,PASS,Sources and references match the sampled valid...
5,Sentence BLEU range,PASS,All sentence scores are finite and between 0 a...
6,Retail rows aligned,PASS,One prediction per retail case with its matchi...
7,Retail scores and timing,PASS,Corpus BLEU is in range and measured request t...
8,Automatic flags consistent,PASS,A case passes only when all four automatic con...
9,Human-review entries,PASS,"Each case has None, True, or False."


All 12 notebook checks passed.
These check notebook consistency, not translation quality or production readiness.
Still review language alignment, duplicate text, translation meaning, and representative retail data.


## Next Steps
Save your observations and continue to `04_model_inference.ipynb`. If you trained a model, load your saved model and tokenizer there to compare translations with the pretrained baseline.